# EmotionSense AI — RC1 Validation (Kaggle)

Validate the **frozen RC1** (`v1.0.0-rc1`) on real corpora. **No source code is changed** —
this notebook only installs the tagged release and runs the prepared experiment configs.

## Prerequisites
1. **Add Data** (right panel) — attach these public Kaggle datasets:
   - `uwrfkaggler/ravdess-emotional-speech-audio`
   - `ejlok1/toronto-emotional-speech-set-tess`
   - `ejlok1/cremad`
2. Settings → **Internet: ON** (needed to clone + pip install).
3. Settings → Accelerator: **GPU** optional (note: RC1's SSL extractor is CPU-bound, so GPU
   won't speed the transformer run — see the transformer cell).

Outputs (leaderboards) are written to `experiments/reports/` and saved as notebook output.

## 1. Install the frozen RC1 release

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/<your-username>/EmotionSenseAI.git'  # <-- SET THIS
GIT_REF  = 'v1.0.0-rc1'  # frozen RC1 tag
if not os.path.isdir('EmotionSenseAI'):
    subprocess.run(['git','clone','--branch',GIT_REF,'--depth','1',REPO_URL], check=True)
os.chdir('EmotionSenseAI')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[transformer,backend]'], check=True)
print('installed at', subprocess.run(['git','describe','--tags'],capture_output=True,text=True).stdout.strip())

## 2. Link the attached Kaggle datasets into `data/raw/`

The frozen loaders `rglob` for `*.wav`, so symlinking each Kaggle input directory is enough.

In [ ]:
import os
from pathlib import Path
Path('data/raw').mkdir(parents=True, exist_ok=True)
CANDIDATES = {
    'ravdess': ['/kaggle/input/ravdess-emotional-speech-audio'],
    'tess':    ['/kaggle/input/toronto-emotional-speech-set-tess'],
    'crema_d': ['/kaggle/input/cremad'],
}
for name, cands in CANDIDATES.items():
    dst = Path('data/raw')/name
    src = next((c for c in cands if os.path.isdir(c)), None)
    if src and not dst.exists():
        os.symlink(src, dst)
    n = len(list(dst.rglob('*.wav'))) if dst.exists() else 0
    print(f'{name:8s} {"OK "+src if src else "MISSING - add the Kaggle dataset"} -> {n} wav')

## 3. CPU classical validation (primary acceptance run)

In [ ]:
# CPU classical validation: baselines + SVM/RF/LogReg, RAVDESS 5-fold + cross-corpus CREMA-D
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_validation.yaml --out-dir experiments/reports

## 4. CREMA-D in-corpus CV

In [ ]:
# CREMA-D in-corpus 5-fold (91 speakers)
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_cremad.yaml --out-dir experiments/reports

## 5. Transformer validation (Distil-HuBERT)

**Runtime note:** RC1 runs SSL extraction on CPU. Expect this to be slow, dominated by the
CREMA-D cross-corpus pass. Reduce `n_folds` in `configs/experiments/rc1_transformer.yaml`
if you only need a quick signal. Do **not** change source code — configs are fair game.

In [ ]:
# Transformer: frozen Distil-HuBERT embeddings + SVM head (SSL extraction is CPU-bound in RC1).
# Embeddings are cached per clip; the CREMA-D cross-corpus pass (~7.4k clips) dominates runtime.
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_transformer.yaml --out-dir experiments/reports

## 6. Results

Copy the printed leaderboards (and the `experiments/reports/*.json`) back into the repo as
the validation record. **Wait for these results before proposing any architectural change.**

In [ ]:
import glob
for md_file in sorted(glob.glob('experiments/reports/*.md')):
    print('='*80); print(md_file); print(open(md_file, encoding='utf-8').read())